# Day 5: Silver Standardization (OMOP CDM v5.4 & Entity Linkage)

This notebook demonstrates probabilistic patient linkage using Splink and PySpark across disparate EHR records (Fellegi-Sunter model), followed by OMOP standardization.

In [ ]:
import os
from pyspark.sql import SparkSession
import splink.spark.comparison_library as cl
from splink.spark.linker import SparkLinker
import pyspark.sql.functions as F

os.environ['SPARK_VERSION'] = '3.3'

spark = SparkSession.builder \
    .appName("Day05_Splink_Linkage") \
    .getOrCreate()

print("SparkSession initialized for Splink Linkage.")

In [ ]:
# Mock data representing records from two different source systems
data = [
    ("1", "John", "Doe", "1980-01-01", "123 Main St", "SysA"),
    ("2", "Jon", "Doe", "1980-01-01", "123 Main Street", "SysB"),
    ("3", "Jane", "Smith", "1990-05-15", "456 Elm St", "SysA"),
    ("4", "Jane", "Smyth", "1990-05-15", "456 Elm St", "SysB")
]
columns = ["unique_id", "first_name", "last_name", "dob", "address", "source_dataset"]

df_patients = spark.createDataFrame(data, columns)
df_patients.show()

In [ ]:
# Define Splink Linkage Settings
settings = {
    "link_type": "dedupe_only",
    "comparisons": [
        cl.exact_match("first_name"),
        cl.exact_match("last_name"),
        cl.exact_match("dob"),
        cl.exact_match("address")
    ],
    "blocking_rules_to_generate_predictions": [
        "l.dob = r.dob",
        "l.first_name = r.first_name"
    ],
    "retain_matching_columns": True,
    "retain_intermediate_calculation_columns": True
}

linker = SparkLinker(df_patients, settings, spark=spark)

# Train models (using simplified default values for demo)
linker.estimate_u_using_random_sampling(max_pairs=1e6)
linker.estimate_parameters_using_expectation_maximisation("l.dob = r.dob")
linker.estimate_parameters_using_expectation_maximisation("l.first_name = r.first_name")

df_predict = linker.predict(threshold_match_probability=0.2)

print("Linkage Predictions:")
df_predict.as_pandas_dataframe().head()

In [ ]:
# Get cluster assignments (Entity Resolution)
df_clusters = linker.cluster_pairwise_predictions_at_threshold(df_predict, 0.5)

print("Resolved Clusters (Assigned Enterprise Person ID):")
df_clusters.as_pandas_dataframe().head()

In [ ]:
# OMOP Transformation Demo
# Using the clusters, we can assign a consistent person_id for the OMOP transformation
# (Assuming we join df_patients and df_clusters to get cluster_id -> person_id)
print("Silver Standardization complete: Entity linkage allows mapping varied EHR source IDs to unified OMOP person_id.")